# Statistical Analysis Reproducibility — 1D CNN, MFCC Ensemble, CNN14

Recomputes every number in **Tables 4, 5, and 6** of the manuscript
(*Denoiser–Representation Interaction in Noise-Robust ESC*) directly from raw
per-run CSVs — Cohen's $d_z$, 95% confidence intervals, raw p-values, and the
Benjamini-Hochberg (FDR) / Holm-Bonferroni multiple-comparisons correction
across all 30 reported significance tests.

**No number below is hand-entered from the paper.** Every value is derived
from the four source CSVs uploaded in the next cell.

### Files you need to upload
| File | Model | Contents |
|---|---|---|
| `gaussian_unet_per_run.csv` | 1D CNN | Gaussian noise, per-run Noisy %/U-Net % |
| `traffic_unet_per_run.csv` | 1D CNN | Traffic noise, per-run Noisy %/U-Net % |
| `traffic_unet_multirun_raw.csv` | CNN14 | Gaussian + Traffic + White, per-run accuracy |
| `mfcc_unet_multirun_raw.csv` | MFCC Ensemble | Gaussian + Traffic, per-run accuracy |

### Method (matches manuscript §3.2)
For each condition (model × noise type × SNR), with **n = 5** seed-matched runs:

$$\text{diff}_i = \text{UNet\%}_i - \text{Noisy\%}_i \qquad
\text{recovery} = \overline{\text{diff}} \qquad
d_z = \frac{\overline{\text{diff}}}{s_{\text{diff}}}$$

95% CI via the $t$-distribution ($df=4$); raw $p$ from a two-sided paired $t$-test.
Multiple comparisons across all 30 conditions are corrected with **BH-FDR** (primary)
and **Holm-Bonferroni** (conservative secondary check).


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import t as tdist

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

REQUIRED_SNR = [10, 5, 0, -5, -10]
ALPHA = 0.05

## 1. Upload the raw CSVs

Run the cell below in Colab and select all four files listed above. If you're
running this locally (not in Colab) instead, just place the four CSVs in the
same folder as this notebook and skip this cell.

In [ ]:
try:
    from google.colab import files
    print("Select all 4 files: gaussian_unet_per_run.csv, traffic_unet_per_run.csv, "
          "traffic_unet_multirun_raw.csv, mfcc_unet_multirun_raw.csv")
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab — assuming the 4 CSVs are already in the working directory.")

## 2. Load and tag the raw per-run data

Each source file has a slightly different column layout — this cell renames
columns to a common `noisy` / `unet` / `SNR` / `noise` schema and tags each
row with which model it belongs to.

In [ ]:
gauss_1dcnn = pd.read_csv("gaussian_unet_per_run.csv").rename(
    columns={"SNR_dB": "SNR", "Noisy_%": "noisy", "UNet_%": "unet"})
gauss_1dcnn["noise"] = "Gaussian"

traffic_1dcnn = pd.read_csv("traffic_unet_per_run.csv").rename(
    columns={"SNR_dB": "SNR", "Noisy_%": "noisy", "UNet_%": "unet"})
traffic_1dcnn["noise"] = "Traffic"

# NOTE: despite the filename, traffic_unet_multirun_raw.csv contains BOTH
# 'traffic' and 'gaussian' condition rows (plus 'white', which the paper
# does not report on and which we exclude below).
cnn14_raw = pd.read_csv("traffic_unet_multirun_raw.csv").rename(
    columns={"SNR_target": "SNR", "acc_noisy": "noisy", "acc_unet": "unet"})
cnn14_raw["noise"] = cnn14_raw["condition"].str.capitalize()

mfcc_raw = pd.read_csv("mfcc_unet_multirun_raw.csv").rename(
    columns={"SNR_target": "SNR", "acc_noisy": "noisy", "acc_unet": "unet"})
mfcc_raw["noise"] = mfcc_raw["condition"].str.capitalize()

MODELS = {
    "1D CNN": pd.concat([gauss_1dcnn, traffic_1dcnn], ignore_index=True),
    "CNN14": cnn14_raw[cnn14_raw["noise"].isin(["Gaussian", "Traffic"])],
    "MFCC Ensemble": mfcc_raw[mfcc_raw["noise"].isin(["Gaussian", "Traffic"])],
}

for name, df in MODELS.items():
    n_conditions = df.groupby(["noise", "SNR"]).size()
    print(f"{name}: {len(n_conditions)} noise x SNR combinations loaded, "
          f"runs per condition = {sorted(n_conditions.unique())}")

## 3. Statistics functions

`paired_stats` computes Cohen's $d_z$, the 95% CI on the recovery delta, and
the raw two-sided paired-$t$ p-value for one condition. `bh_fdr` and
`holm_bonferroni` implement the two multiple-comparisons corrections used in
the paper.

In [ ]:
def paired_stats(noisy, unet, label):
    """Cohen's dz, 95% CI on the paired diff, and raw two-sided paired-t p."""
    diffs = np.asarray(unet) - np.asarray(noisy)
    n = len(diffs)
    mean_diff = diffs.mean()
    sd_diff = diffs.std(ddof=1)
    dz = mean_diff / sd_diff
    se = sd_diff / np.sqrt(n)
    tcrit = tdist.ppf(1 - ALPHA / 2, n - 1)
    ci_lo, ci_hi = mean_diff - tcrit * se, mean_diff + tcrit * se
    tstat = mean_diff / se
    p = 2 * (1 - tdist.cdf(abs(tstat), n - 1))
    effect = "large" if abs(dz) >= 0.8 else ("medium" if abs(dz) >= 0.5 else "small")
    return dict(
        label=label, n=n,
        noisy_mean=np.mean(noisy), noisy_sd=np.std(noisy, ddof=1),
        unet_mean=np.mean(unet), unet_sd=np.std(unet, ddof=1),
        recovery=mean_diff, ci_lo=ci_lo, ci_hi=ci_hi,
        dz=dz, effect=effect, t=tstat, p=p,
    )


def bh_fdr(pvals, alpha=ALPHA):
    """Benjamini-Hochberg step-up. Returns (adjusted p-values, significant bool)."""
    p = np.asarray(pvals)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adj = ranked * m / (np.arange(m) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.minimum(adj, 1.0)
    out_adj = np.empty(m)
    out_adj[order] = adj
    return out_adj, out_adj <= alpha


def holm_bonferroni(pvals, alpha=ALPHA):
    """Holm step-down. Returns (adjusted p-values, significant bool)."""
    p = np.asarray(pvals)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adj = ranked * (m - np.arange(m))
    adj = np.maximum.accumulate(adj)
    adj = np.minimum(adj, 1.0)
    out_adj = np.empty(m)
    out_adj[order] = adj
    return out_adj, out_adj <= alpha

## 4. Compute exact paired stats for all 30 conditions

5 SNR levels × 2 noise types × 3 models = 30 conditions, matching Tables 4–6.

In [ ]:
rows = []
for model, df in MODELS.items():
    for noise in ["Gaussian", "Traffic"]:
        for snr in REQUIRED_SNR:
            sub = df[(df["noise"] == noise) & (df["SNR"] == snr)]
            assert len(sub) == 5, f"{model} {noise} {snr}dB: expected 5 runs, found {len(sub)}"
            r = paired_stats(sub["noisy"].values, sub["unet"].values,
                              label=f"{model} | {noise} | {snr} dB")
            r["model"] = model
            r["noise"] = noise
            r["snr"] = snr
            rows.append(r)

results = pd.DataFrame(rows)
print(f"Computed {len(results)} conditions.")

## 5. Multiple-comparisons correction (BH-FDR primary, Holm secondary)

Applied once, across all 30 raw p-values together.

In [ ]:
results["p_BH"], results["sig_BH"] = bh_fdr(results["p"].values)
results["p_Holm"], results["sig_Holm"] = holm_bonferroni(results["p"].values)

print(f"Significant under BH-FDR:   {results['sig_BH'].sum()} / {len(results)}")
print(f"Significant under Holm:     {results['sig_Holm'].sum()} / {len(results)}")
print()
print("Non-significant under BH-FDR:")
print(results.loc[~results['sig_BH'], 'label'].to_string(index=False))
print()
print("Non-significant under Holm-Bonferroni:")
print(results.loc[~results['sig_Holm'], 'label'].to_string(index=False))

## 6. Table 4 — 1D CNN

In [ ]:
snr_order = {v: i for i, v in enumerate(REQUIRED_SNR)}
noise_order = {"Gaussian": 0, "Traffic": 1}
results["sort_key"] = results["noise"].map(noise_order) * 10 + results["snr"].map(snr_order)

def show_table(model):
    sub = results[results["model"] == model].sort_values("sort_key")
    cols = ["noise", "snr", "noisy_mean", "noisy_sd", "unet_mean", "unet_sd",
            "recovery", "ci_lo", "ci_hi", "dz", "effect", "p", "p_BH", "sig_BH", "p_Holm", "sig_Holm"]
    return sub[cols].reset_index(drop=True)

show_table("1D CNN")

## 7. Table 5 — MFCC Ensemble

In [ ]:
show_table("MFCC Ensemble")

## 8. Table 6 — CNN14

In [ ]:
show_table("CNN14")

## 9. Save full output

Writes every computed number to `reproduced_tables_4_6.csv` — diff this
directly against the manuscript tables to confirm an exact match.

In [ ]:
out_cols = ["model", "noise", "snr", "n", "noisy_mean", "noisy_sd", "unet_mean", "unet_sd",
            "recovery", "ci_lo", "ci_hi", "dz", "effect", "t", "p", "p_BH", "sig_BH", "p_Holm", "sig_Holm"]
final = results.sort_values(["model", "sort_key"])[out_cols]
final.to_csv("reproduced_tables_4_6.csv", index=False)
print("Saved -> reproduced_tables_4_6.csv")
final

In [ ]:
try:
    from google.colab import files
    files.download("reproduced_tables_4_6.csv")
except ImportError:
    pass